In [ ]:
import pandas as pd
import numpy as np

# Load the cleaned dataset from Part 1
df = pd.read_csv("cleaned_data.csv")
print(f"Loaded shape: {df.shape}")

# Cap outliers in SalePrice and Gr Liv Area (limits extreme values, keeps all rows)
def cap_iqr(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    return series.clip(lower=lower, upper=upper)

for col in ["SalePrice", "Gr Liv Area"]:
    before_max = df[col].max()
    df[col] = cap_iqr(df[col])
    print(f"{col}: capped. max before={before_max:.2f}, max after={df[col].max():.2f}")

# Regression target: predict the actual price
y_reg = df["SalePrice"]

# Classification target: is this house above or below the median price?
y_clf = (y_reg > y_reg.median()).astype(int)

print(f"\ny_reg stats:\n{y_reg.describe()}")
print(f"\ny_clf class balance:\n{y_clf.value_counts(normalize=True)}")

# Feature matrix: everything except the target and the two ID columns
drop_cols = ["SalePrice", "Order", "PID"]
X = df.drop(columns=drop_cols)
print(f"\nFeature matrix X shape: {X.shape}")
print(f"X columns: {X.columns.tolist()}")

In [ ]:
# STEP 1: Handle "None means the feature doesn't exist" columns
none_means_no_feature = ["Pool QC", "Alley", "Fence", "Misc Feature",
                          "Fireplace Qu", "Mas Vnr Type", "Garage Type",
                          "Garage Finish", "Garage Qual", "Garage Cond",
                          "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
                          "BsmtFin Type 1", "BsmtFin Type 2"]

for col in none_means_no_feature:
    X[col] = X[col].fillna("None")

# STEP 2: Fill remaining categorical nulls with the mode
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")

remaining_nulls = X[cat_cols].isnull().sum()
print(f"\nRemaining categorical nulls before mode-fill:\n{remaining_nulls[remaining_nulls > 0]}")

for col in cat_cols:
    if X[col].isnull().sum() > 0:
        X[col] = X[col].fillna(X[col].mode()[0])

print(f"\nCategorical nulls after fill: {X[cat_cols].isnull().sum().sum()}")

# STEP 3: LABEL ENCODING for ordinal quality/condition columns (natural order)
ordinal_maps = {
    "Exter Qual":   {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Exter Cond":   {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Bsmt Qual":    {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Bsmt Cond":    {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Heating QC":   {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Kitchen Qual": {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Fireplace Qu": {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Garage Qual":  {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Garage Cond":  {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "Pool QC":      {"None": 0, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
}

for col, mapping in ordinal_maps.items():
    print(f"\n{col} raw values: {X[col].unique()}")
    X[col + "_encoded"] = X[col].map(mapping)
    unmapped = X.loc[X[col + "_encoded"].isna(), col].unique()
    print(f"{col} unmapped values (should be empty): {unmapped}")
    X = X.drop(columns=[col])

# STEP 4: ONE-HOT ENCODING for nominal columns (no natural order)
nominal_cols = ["MS Zoning", "Street", "Alley", "Lot Shape", "Land Contour",
                 "Utilities", "Lot Config", "Land Slope", "Neighborhood",
                 "Condition 1", "Condition 2", "Bldg Type", "House Style",
                 "Roof Style", "Roof Matl", "Exterior 1st", "Exterior 2nd",
                 "Mas Vnr Type", "Foundation", "Bsmt Exposure",
                 "BsmtFin Type 1", "BsmtFin Type 2", "Heating", "Central Air",
                 "Electrical", "Functional", "Garage Type", "Garage Finish",
                 "Paved Drive", "Fence", "Misc Feature", "Sale Type",
                 "Sale Condition", "MS SubClass"]

X = pd.get_dummies(X, columns=nominal_cols, drop_first=True)

print(f"\nFinal X shape after encoding: {X.shape}")
print(f"Remaining nulls in X: {X.isnull().sum().sum()}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split X and both labels together (same random_state -> same rows in same split)
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_reg_train shape: {y_reg_train.shape}, y_reg_test shape: {y_reg_test.shape}")
print(f"y_clf_train shape: {y_clf_train.shape}, y_clf_test shape: {y_clf_test.shape}")

# Fit the scaler ONLY on training features - never on test or full data
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nX_train_scaled mean (should be ~0): {X_train_scaled.mean():.4f}")
print(f"X_train_scaled std (should be ~1): {X_train_scaled.std():.4f}")
print(f"X_test_scaled mean: {X_test_scaled.mean():.4f}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Unconstrained Decision Tree (no depth limit)
dt_unconstrained = DecisionTreeClassifier(random_state=42)
dt_unconstrained.fit(X_train_scaled, y_clf_train)

train_acc_unconstrained = accuracy_score(y_clf_train, dt_unconstrained.predict(X_train_scaled))
test_acc_unconstrained = accuracy_score(y_clf_test, dt_unconstrained.predict(X_test_scaled))

print("Unconstrained Decision Tree:")
print(f"  Training accuracy = {train_acc_unconstrained:.4f}")
print(f"  Test accuracy     = {test_acc_unconstrained:.4f}")
print(f"  Train-test gap    = {train_acc_unconstrained - test_acc_unconstrained:.4f}")

In [ ]:
# Controlled Decision Tree (max_depth=5, min_samples_split=20)
dt_controlled = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_controlled.fit(X_train_scaled, y_clf_train)

train_acc_controlled = accuracy_score(y_clf_train, dt_controlled.predict(X_train_scaled))
test_acc_controlled = accuracy_score(y_clf_test, dt_controlled.predict(X_test_scaled))

print("Controlled Decision Tree (max_depth=5, min_samples_split=20):")
print(f"  Training accuracy = {train_acc_controlled:.4f}")
print(f"  Test accuracy     = {test_acc_controlled:.4f}")
print(f"  Train-test gap    = {train_acc_controlled - test_acc_controlled:.4f}")

print(f"\nComparison:")
print(f"  Unconstrained gap = {train_acc_unconstrained - test_acc_unconstrained:.4f}")
print(f"  Controlled gap    = {train_acc_controlled - test_acc_controlled:.4f}")

In [ ]:
# Gini vs Entropy comparison (both with max_depth=5)
dt_gini = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
dt_gini.fit(X_train_scaled, y_clf_train)
test_acc_gini = accuracy_score(y_clf_test, dt_gini.predict(X_test_scaled))

dt_entropy = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42)
dt_entropy.fit(X_train_scaled, y_clf_train)
test_acc_entropy = accuracy_score(y_clf_test, dt_entropy.predict(X_test_scaled))

print("Gini vs Entropy comparison (max_depth=5):")
print(f"  Gini criterion test accuracy    = {test_acc_gini:.4f}")
print(f"  Entropy criterion test accuracy = {test_acc_entropy:.4f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd

rf_clf_p3 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_clf_p3.fit(X_train_scaled, y_clf_train)

train_acc_rf = accuracy_score(y_clf_train, rf_clf_p3.predict(X_train_scaled))
test_acc_rf = accuracy_score(y_clf_test, rf_clf_p3.predict(X_test_scaled))
auc_rf_p3 = roc_auc_score(y_clf_test, rf_clf_p3.predict_proba(X_test_scaled)[:, 1])

print("Random Forest (n_estimators=100, max_depth=10):")
print(f"  Training accuracy = {train_acc_rf:.4f}")
print(f"  Test accuracy     = {test_acc_rf:.4f}")
print(f"  ROC-AUC           = {auc_rf_p3:.4f}")

# Top 5 features by importance
importance_df_p3 = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_clf_p3.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 5 features by importance:")
print(importance_df_p3.head(5).to_string(index=False))

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_clf.fit(X_train_scaled, y_clf_train)

train_acc_gb = accuracy_score(y_clf_train, gb_clf.predict(X_train_scaled))
test_acc_gb = accuracy_score(y_clf_test, gb_clf.predict(X_test_scaled))
auc_gb = roc_auc_score(y_clf_test, gb_clf.predict_proba(X_test_scaled)[:, 1])

print("Gradient Boosting (n_estimators=100, learning_rate=0.1, max_depth=3):")
print(f"  Training accuracy = {train_acc_gb:.4f}")
print(f"  Test accuracy     = {test_acc_gb:.4f}")
print(f"  ROC-AUC           = {auc_gb:.4f}")

In [ ]:
import numpy as np

# Identify 5 lowest-importance features from the Random Forest (Task 4)
lowest_5_features = importance_df_p3.tail(5)["feature"].tolist()
print(f"5 lowest-importance features: {lowest_5_features}")

# Full model AUC (already computed in Task 4)
auc_full = auc_rf_p3
print(f"\nFull model (all features) test AUC: {auc_full:.4f}")

# Reduced model: same RF hyperparameters, minus the 5 lowest-importance features
lowest_5_indices = [X.columns.get_loc(f) for f in lowest_5_features]
X_train_reduced = np.delete(X_train_scaled, lowest_5_indices, axis=1)
X_test_reduced = np.delete(X_test_scaled, lowest_5_indices, axis=1)

rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reduced.fit(X_train_reduced, y_clf_train)

auc_reduced = roc_auc_score(y_clf_test, rf_reduced.predict_proba(X_test_reduced)[:, 1])
print(f"Reduced model (5 lowest-importance features removed) test AUC: {auc_reduced:.4f}")

print(f"\nAUC difference (full - reduced): {auc_full - auc_reduced:.4f}")

In [ ]:
# OPTIONAL bonus experiment - NOT replacing the required Task 4 output
rf_shallow = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_shallow.fit(X_train_scaled, y_clf_train)

train_acc_shallow = accuracy_score(y_clf_train, rf_shallow.predict(X_train_scaled))
test_acc_shallow = accuracy_score(y_clf_test, rf_shallow.predict(X_test_scaled))

print("Bonus: Random Forest with reduced max_depth=6 (optional exploration):")
print(f"  Training accuracy = {train_acc_shallow:.4f}")
print(f"  Test accuracy     = {test_acc_shallow:.4f}")
print(f"  Gap               = {train_acc_shallow - test_acc_shallow:.4f}")

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
import pandas as pd

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_to_compare = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree (max_depth=5)": DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
}

cv_results = []
for name, model in models_to_compare.items():
    scores = cross_val_score(model, X_train_scaled, y_clf_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results.append({"Model": name, "Mean AUC": scores.mean(), "Std AUC": scores.std()})
    print(f"{name}: mean AUC = {scores.mean():.4f}, std = {scores.std():.4f}")

cv_df = pd.DataFrame(cv_results)
print("\nCross-validated comparison table:")
print(cv_df.to_string(index=False))

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV

param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}

pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

grid_search = GridSearchCV(
    pipeline, param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc', n_jobs=-1
)

# Note: unscaled X_train - the pipeline handles imputation + scaling internally
grid_search.fit(X_train, y_clf_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best score (mean CV AUC): {grid_search.best_score_:.4f}")

best_pipeline = grid_search.best_estimator_

In [ ]:
from sklearn.metrics import roc_auc_score
import pandas as pd

fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
learning_curve_results = []

for f in fractions:
    n_rows = int(f * len(X_train))
    X_subset = X_train.iloc[:n_rows]
    y_subset = y_clf_train.iloc[:n_rows]

    # Fit a fresh copy of the best pipeline on this subset
    from sklearn.base import clone
    pipeline_subset = clone(best_pipeline)
    pipeline_subset.fit(X_subset, y_subset)

    train_auc = roc_auc_score(y_subset, pipeline_subset.predict_proba(X_subset)[:, 1])
    test_auc = roc_auc_score(y_clf_test, pipeline_subset.predict_proba(X_test)[:, 1])

    learning_curve_results.append({"Training fraction": f, "Training AUC": train_auc, "Test AUC": test_auc})
    print(f"Fraction={f}: Train AUC={train_auc:.4f}, Test AUC={test_auc:.4f}")

lc_df = pd.DataFrame(learning_curve_results)
print("\nLearning curve table:")
print(lc_df.to_string(index=False))

In [ ]:
import joblib

joblib.dump(best_pipeline, 'best_model.pkl')
print("Saved best_model.pkl")

# Reload and predict on two hand-crafted test rows
loaded_model = joblib.load('best_model.pkl')

# Two rows from the actual test set as "hand-crafted" example inputs
sample_rows = X_test.iloc[:2]
predictions = loaded_model.predict(sample_rows)
probabilities = loaded_model.predict_proba(sample_rows)[:, 1]

print(f"\nSample predictions: {predictions}")
print(f"Sample probabilities: {probabilities}")
print("Reload-and-predict successful, no errors.")

from google.colab import files
files.download('best_model.pkl')

In [ ]:
from sklearn.metrics import roc_auc_score

# Recompute Logistic Regression AUC in THIS session (auc_baseline didn't exist here)
log_reg_p3 = LogisticRegression(max_iter=1000, random_state=42)
log_reg_p3.fit(X_train_scaled, y_clf_train)
auc_logreg = roc_auc_score(y_clf_test, log_reg_p3.predict_proba(X_test_scaled)[:, 1])

# Decision Tree test AUC (dt_controlled already trained earlier this session)
auc_dt = roc_auc_score(y_clf_test, dt_controlled.predict_proba(X_test_scaled)[:, 1])

# Tuned RF (best_pipeline) test AUC
auc_tuned_rf = roc_auc_score(y_clf_test, best_pipeline.predict_proba(X_test)[:, 1])

summary_data = {
    "Logistic Regression": {"CV Mean AUC": 0.9685, "CV Std AUC": 0.0046, "Test AUC": auc_logreg},
    "Decision Tree (max_depth=5)": {"CV Mean AUC": 0.9279, "CV Std AUC": 0.0035, "Test AUC": auc_dt},
    "Random Forest": {"CV Mean AUC": 0.9794, "CV Std AUC": 0.0044, "Test AUC": auc_rf_p3},
    "Gradient Boosting": {"CV Mean AUC": 0.9812, "CV Std AUC": 0.0032, "Test AUC": auc_gb},
    "Tuned RF (GridSearchCV)": {"CV Mean AUC": grid_search.best_score_, "CV Std AUC": None, "Test AUC": auc_tuned_rf},
}

summary_df = pd.DataFrame(summary_data).T
print(summary_df)